# 🧩 12 · Clustering: k-means, hierarchical clustering and co-expressed genes
### Finding natural groups without a teacher — distances, the k-means algorithm, dendrograms and class discovery in leukaemia

**BiologyToML** · Part C — Unsupervised learning · Module 12 of 24

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/ml/tutorial/BiologyToML/notebooks/12_clustering_kmeans_hierarchical.ipynb)

| | |
|---|---|
| **Hardware target** | Google Colab free tier — CPU only (≈ 1 min to run everything); no GPU needed |
| **Core frameworks** | Python 3, NumPy, SciPy (`scipy.cluster.hierarchy`, for checking), pandas, Matplotlib, scikit-learn (ARI and checks only) |
| **Prerequisites** | Modules 01–05; [Module 03](03_linear_algebra_for_data.ipynb) (vectors, distances) and [Module 08](08_knn_naive_bayes.ipynb) (distance-based learning) |
| **Estimated time** | 150 min (reading + running + the ✏️ exercises) |
| **02-620 lectures covered** | `006_clustering` slides 1–57 (+ a preview of `010_mixture-model_mz_031725` slides 3 and 78) — see [`SLIDE_MAP.md`](../SLIDE_MAP.md) |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis.
> **Real data:** the Golub *et al.* (1999) leukaemia expression matrix (72 patients × 3,571 genes) in Section 8.
> **Simulated data:** the 2-D "expression in two conditions" point clouds and the yeast-like cell-cycle time course
> are SIMULATED (every plot title says so); the two 5-object distance matrices are copied from the 02-620 slides.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **Explain** the difference between supervised and unsupervised learning, and what it means to cluster samples, genes, or both.
2. **Compute** Euclidean and correlation distances by hand and in code, and **choose** between them for a biological question.
3. **Implement** the k-means algorithm from scratch, **prove** that each step cannot increase its objective, and **diagnose** local optima with restarts.
4. **Use** the elbow of the k-means objective to suggest a number of clusters — and **recognise** when the elbow is uninformative.
5. **Carry out** single-, complete- and average-linkage agglomerative clustering by hand on the slides' distance matrices, then **implement** it and **check** it against SciPy.
6. **Read** a dendrogram: merge heights, cuts, chaining, outliers.
7. **Apply** both methods to real leukaemia data, **evaluate** the clusters against ALL/AML with a contingency table and the adjusted Rand index, and **interpret** how much the answer depends on analysis choices.

## 🗺️ Contents

| Section | What you will do | ⏱ |
|---|---|---|
| 0 · Setup | imports, toolkit, data loader | 5 min |
| 1 · Learning without labels | supervised vs unsupervised; clustering samples vs genes | 10 min |
| 2 · Similarity and distance | Euclidean vs correlation; when to use which | 15 min |
| 3 · A simulated yeast cell-cycle experiment | 300 genes × 12 time points, 4 programmes (SIMULATED) | 10 min |
| 4 · k-means | objective, algorithm from scratch, why it converges, local optima | 30 min |
| 5 · How many clusters? | the elbow — clear toy vs real data | 15 min |
| 6 · Hierarchical clustering | the slides' matrices by hand; linkages; implementation; chaining | 30 min |
| 7 · Properties of hierarchical clustering | cost, cuts, outliers | 10 min |
| 8 · Real data: class discovery in leukaemia | Golub ALL/AML without labels; heatmap; sensitivity | 20 min |
| ✅ / 🧠 / 🚀 | checkpoint, discussion, going further | 5 min |

## 🧫 The biological question

In 1998 Michael Eisen, Patrick Brown, David Botstein and colleagues measured the expression of thousands of yeast genes
over time and did something simple but powerful: they **grouped genes whose expression went up and down together**
(Eisen *et al.* 1998). Genes in the same group turned out to share functions — ribosome biogenesis, DNA replication,
the cell cycle. The logic, often called **"guilt by association"**, is still everywhere in genomics: if an uncharacterised
gene is co-expressed with known DNA-replication genes, DNA replication is a good first hypothesis for what it does.
Slide 5 of the lecture shows a human gene co-expression network built on exactly this idea.

The same tool can group **patients** instead of genes. A year later, Golub *et al.* (1999) asked whether the two main
types of acute leukaemia — **acute lymphoblastic (ALL)** and **acute myeloid (AML)** — could be *rediscovered* from
expression profiles alone, without telling the computer which patient had which disease. They called this
**class discovery** (the title of their paper), as opposed to **class prediction** (Modules 08–11). Perou *et al.* (2000)
later used clustering to define the "intrinsic" subtypes of breast cancer (luminal, basal-like, HER2-enriched, …) that
remain central to breast-cancer research and diagnostics.

In none of these studies was there a "right answer" column to learn from. That is **unsupervised learning**, and
clustering is its most common form. In this notebook you will build the two classical clustering methods of 02-620 —
**k-means** and **hierarchical clustering** — from scratch, run them on a simulated cell-cycle experiment, and then
try Golub's class-discovery experiment yourself on the real data.

---
## 0 · Environment setup

Run the cells **in order, from the top** (Shift + Enter). The first cell reports your hardware; this module needs only a CPU.

In [ ]:
# ── Hardware check ──
import platform
import subprocess
import sys

print("Python", sys.version.split()[0], "on", platform.system())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", gpu or "none detected — that is fine, a CPU is enough for this course.")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU: none detected — that is fine, a CPU is enough for this course.")

In [ ]:
# ── Dependencies (Colab already has these; we install only what is missing) ──
import importlib

for module, pip_name in [("numpy", "numpy"), ("scipy", "scipy"), ("pandas", "pandas"), ("matplotlib", "matplotlib"), ("sklearn", "scikit-learn")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
print("✅ All dependencies are available.")

In [ ]:
# ── Global imports and configuration ──
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import ListedColormap
from scipy.cluster import hierarchy
from scipy.spatial.distance import pdist, squareform
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

SEED = 0                                   # fixing the random seed makes every run identical
rng = np.random.default_rng(SEED)
plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6.4, 3.6), "axes.titlesize": 10,
                     "axes.labelsize": 9, "legend.fontsize": 8, "axes.spines.top": False,
                     "axes.spines.right": False})
print("NumPy", np.__version__, "| seed", SEED)

### 🧰 Course toolkit

The next cell copies the shared helpers this notebook needs from [`course_toolkit.py`](../course_toolkit.py).
`load_golub` reads the real leukaemia data (with a clearly labelled simulated fallback if you are offline), `standardize` z-scores columns and `CLASS_COLORS` is the course's colour-blind-safe palette. Just run it.

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v1.0; copied so this notebook runs alone) ──
import os
import urllib.request
import numpy as np
import pandas as pd

REPO_DATA_URL = "https://raw.githubusercontent.com/xulabs/edu/main/ml/tutorial/BiologyToML/datasets/{name}"


CACHE_DIR = os.environ.get("BIO2ML_CACHE", "course_data")


def fetch_course_file(name: str, timeout: float = 60.0) -> str | None:
    """Return a local path to the course data file ``name``, or ``None`` if it cannot be found.

    Looks in the repository's ``datasets/`` folder first (when the notebook runs inside a
    clone of xulabs/edu), then in the download cache, and finally downloads it from GitHub.
    """
    for base in ("../datasets", "datasets", "ml/tutorial/BiologyToML/datasets", CACHE_DIR):
        path = os.path.join(base, name)
        if os.path.exists(path) and os.path.getsize(path) > 0:
            return path
    dest = os.path.join(CACHE_DIR, name)
    os.makedirs(CACHE_DIR, exist_ok=True)
    try:
        with urllib.request.urlopen(REPO_DATA_URL.format(name=name), timeout=timeout) as r, \
                open(dest + ".part", "wb") as f:
            f.write(r.read())
        os.replace(dest + ".part", dest)
        return dest
    except Exception as e:  # noqa: BLE001 — any network problem → offline fallback
        print(f"⚠️  Could not obtain {name} ({type(e).__name__}); using a simulated stand-in instead.")
        return None


def simulate_golub_like(rng: np.random.Generator) -> dict:
    """Simulated stand-in for the Golub data: 72 samples (47 'ALL', 25 'AML') × 3,571 genes.

    Log10-scale expression with gene-specific means, correlated gene modules and
    60 genes that differ between the classes.  Same keys as ``load_golub``.
    """
    n_all, n_aml, J = 47, 25, 3571
    y = np.r_[np.zeros(n_all, int), np.ones(n_aml, int)]
    base = rng.normal(2.6, 0.45, J)
    modules = rng.normal(0, 1, (len(y), 20))                  # 20 hidden "programmes"
    loading = rng.normal(0, 0.15, (20, J)) * (rng.random((20, J)) < 0.1)
    X = base + modules @ loading + rng.normal(0, 0.25, (len(y), J))
    informative = rng.choice(J, 60, replace=False)
    X[:, informative] += np.outer(y - 0.5, rng.choice([-1, 1], 60) * rng.uniform(0.4, 0.9, 60))
    is_train = np.r_[np.arange(n_all) < 27, np.arange(n_aml) < 11]
    return dict(X=X, y=y, labels=np.where(y == 1, "AML", "ALL"), genes=np.array([f"gene{j:04d}" for j in range(J)]),
                is_train=is_train, sample_id=np.arange(1, len(y) + 1), source="SIMULATED stand-in (offline)")


def load_golub() -> dict:
    """Golub et al. (1999) leukaemia expression data, preprocessed as in Dudoit et al. (2002).

    Returns a dict with ``X`` (72 × 3,571, log10 expression), ``y`` (0 = ALL, 1 = AML),
    ``labels``, ``genes`` (Affymetrix probe IDs), ``is_train`` (the original 38/34 split),
    ``sample_id`` and ``source``.  Falls back to ``simulate_golub_like`` when offline.

    Raw intensities were clipped to [100, 16000] before the log10, so the value 2.0 means
    "at or below the detection limit" (about 20 % of all entries).  Some genes are therefore
    constant within a class — per-class variance estimates need a small floor.
    """
    path = fetch_course_file("golub_leukemia.csv.gz")
    if path is None:
        return simulate_golub_like(np.random.default_rng(1999))
    df = pd.read_csv(path)
    return dict(X=df.iloc[:, 3:].to_numpy(float), y=(df["label"] == "AML").to_numpy(int),
                labels=df["label"].to_numpy(), genes=np.array(df.columns[3:]),
                is_train=(df["golub_split"] == "train").to_numpy(), sample_id=df["sample_id"].to_numpy(),
                source="Golub et al. 1999 (real data)")


def standardize(X, mean=None, sd=None):
    """Z-score the columns of X. Pass the training mean/sd to transform test data the same way.

    Returns ``(Z, mean, sd)``; constant columns get sd = 1 so they map to 0.
    """
    X = np.asarray(X, float)
    mean = X.mean(axis=0) if mean is None else mean
    sd = X.std(axis=0) if sd is None else sd
    sd = np.where(sd > 0, sd, 1.0)
    return (X - mean) / sd, mean, sd


CLASS_COLORS = ["#1f77b4", "#d62728", "#2ca02c", "#9467bd", "#ff7f0e", "#8c564b"]


print("🧰 Toolkit loaded:", ", ".join(['fetch_course_file', 'simulate_golub_like', 'load_golub', 'standardize']))

---
## 1 · Learning without labels

> 📍 **Where we are.** Modules 06–11 were all **supervised**: every training sample came with an answer $y$ — a
> quantitative trait, "cancer / healthy", "ALL / AML" — and the job was to learn the map from $X$ to $y$. Now the teacher
> leaves the room. We keep the data matrix $X$ but there is no $y$ column. What can we still learn?

The slides (2–4) put the two situations side by side:

| | Supervised learning | Unsupervised learning |
|---|---|---|
| Training data | pairs $(X, Y)$: inputs **and** labels / outputs | $X$ only |
| Typical goal | predict $y$ for a new $x$ | find structure: groups, low-dimensional patterns |
| How to check | error on held-out labelled data | much harder — there is no answer key |

**Clustering** (slide 12) means organising objects into groups — *clusters* — so that there is **high similarity within
a cluster** and **low similarity between clusters**: "finding natural groupings among objects".

🧮 **Math refresher — the data matrix.** As everywhere in this course, data form a matrix $X$ with **N rows = samples**
(patients, cells, time points) and **J columns = features** (genes, proteins). Sample $i$ is the row vector
$\mathbf x_i = (x_{i1}, \dots, x_{iJ})$. The clustering slides call the number of features $p$ (slide 17: "two objects
$x$ and $y$ both have $p$ features"); we write $J$ to stay consistent with the rest of the course — it is the same number.

In [ ]:
# ── 1.1 The same measurements seen with and without labels (SIMULATED) ──
rng = np.random.default_rng(SEED + 1)
centres_true = np.array([[4.1, 4.0], [0.8, 1.4], [3.8, 1.2]])        # three cell states (made up)
spreads = [0.45, 0.55, 0.65]
X_toy = np.vstack([rng.normal(c, s, size=(20, 2)) for c, s in zip(centres_true, spreads)])
y_toy = np.repeat([0, 1, 2], 20)                                        # the "labels" — hidden in real life

fig, axes = plt.subplots(1, 2, figsize=(9, 3.6), sharex=True, sharey=True)
for c in range(3):
    axes[0].scatter(*X_toy[y_toy == c].T, color=CLASS_COLORS[c], s=22, edgecolor="k", lw=0.3, label=f"class {c}")
axes[0].set_title("supervised view: every sample has a label (SIMULATED)")
axes[0].legend(loc="lower right")
axes[1].scatter(*X_toy.T, color="0.3", s=22)
axes[1].set_title("unsupervised view: same points, no labels")
for ax in axes:
    ax.set_xlabel("expression in condition 1 (log scale)")
axes[0].set_ylabel("expression in condition 2 (log scale)")
plt.tight_layout(); plt.show()
print("data matrix shape (N samples, J features):", X_toy.shape)

🔍 **What just happened?** The left panel is what a supervised learner sees: 60 samples, each with a colour. The right
panel is what a clustering algorithm sees: the *same* 60 points in grey. Your eye still finds three groups, because the
points are bunched — that intuition of "bunched together" is what we must turn into an algorithm, which starts with
making "similar" precise (Section 2).

### Clustering samples, genes — or both

The slides (8–11) draw the data matrix and point out that we can cluster along either axis:

- **cluster the rows (samples)** — e.g. patients into cancer subtypes: each patient is an object described by $J$ genes;
- **cluster the columns (features)** — e.g. genes into co-expression modules: each gene is an object described by its
  $N$ measurements (the matrix is transposed, $X^\top$);
- **cluster both** — the classic expression *heatmap* with a tree on each side (slides 6, 7 and 56; Section 8 below).

Slide 14 then lists the methods of this part of the course: **non-probabilistic** clustering — *hierarchical clustering*
and the *k-means algorithm* (this notebook) — and **probabilistic** clustering with *mixture models*
([Module 13](13_mixture_models_em.ipynb)). Dimensionality reduction ([Module 14](14_pca_dimensionality_reduction.ipynb))
is the other unsupervised workhorse.

In [ ]:
# ── 1.2 One matrix, two clustering problems: Golub leukaemia data ──
golub = load_golub()
print("source:", golub["source"])
X_golub = golub["X"]
print(f"cluster SAMPLES: {X_golub.shape[0]} objects (patients), each described by {X_golub.shape[1]} features (genes)")
print(f"cluster GENES  : {X_golub.T.shape[0]} objects (genes), each described by {X_golub.T.shape[1]} features (patients)")

🔍 **What just happened?** One real data matrix gives two very different clustering problems: 72 points in a
3,571-dimensional space, or 3,571 points in a 72-dimensional space. We return to both in Section 8.

✏️ **Your turn.** In cell 1.1, move the third centre from `[3.8, 1.2]` to `[3.8, 2.6]` and re-run. At what point do you
stop seeing three groups in the right-hand panel? Clustering can only find structure that is actually there.

### 📑 Slide connection — `006_clustering`, slides 2–14

- **What the slides show.** Supervised learning uses (X, Y); unsupervised learning uses X only, "without a teacher" (slides 2–4). Biological
    examples: a human gene co-expression network (5), muscle-tissue expression with two clusters (6), and a heatmap of genes
    expressed in different parts of the lotus plant (7; the gene names *Lj…* indicate the model legume *Lotus japonicus*).
    The data matrix can be clustered by features, by samples, or both (8–11). Clustering = high intra-cluster and low
    inter-cluster similarity (12); further examples (people, flags, words, species; 13); the method map — hierarchical and
    k-means (non-probabilistic) vs mixture models (probabilistic) (14).
- **What this notebook adds.** A side-by-side labelled/unlabelled view of the same simulated cloud, the course's N × J convention (the slides
    write *p* for the number of features), and the two transposed clustering problems hidden in the real Golub matrix.
- **Exam tip.** Be ready to say which axis of X you cluster for a given question ("find cancer subtypes" → rows/samples;
    "find co-regulated genes" → columns/genes) and why clustering has no single correct answer to check against.

---
## 2 · Similarity and distance

> 📍 **Where we are.** "High similarity within clusters" (Section 1) is meaningless until we say how similar two objects
> are. You met distances in [Module 03](03_linear_algebra_for_data.ipynb) and used them for k-nearest neighbours in
> [Module 08](08_knn_naive_bayes.ipynb). Here we compare the two measures on slide 17 and see that the choice can
> completely change which genes look alike.

Slide 15 admits that similarity is "hard to define, but we know it when we see it" (a woman and a spaniel with the same
curly hair!) and takes a pragmatic approach. Slide 18 defines a **distance (dissimilarity)** $D(O_1, O_2)$ as a real
number assigned to every pair of objects — a gorilla and a chimpanzee, two news articles, two genes. Useful distances
also satisfy: $D \ge 0$; $D(O, O) = 0$; symmetry $D(O_1, O_2) = D(O_2, O_1)$; and often the triangle inequality
$D(O_1, O_3) \le D(O_1, O_2) + D(O_2, O_3)$.

🧮 **Math refresher — sums, means and standard deviations.** For a vector $\mathbf x = (x_1, \dots, x_J)$, the symbol
$\sum_{j=1}^{J} a_j$ means "add up $a_1 + a_2 + \dots + a_J$". The **mean** is $\mu_x = \frac1J\sum_j x_j$ and the
**standard deviation** is $\sigma_x = \sqrt{\frac1J\sum_j (x_j-\mu_x)^2}$ — the typical distance of an entry from the mean.

**Euclidean distance** (slide 17) — the straight-line ruler distance, Pythagoras in $J$ dimensions:

$$d(\mathbf x, \mathbf y) = \sqrt{\sum_{j=1}^{J} (x_j - y_j)^2}.$$

*In words:* subtract the two profiles entry by entry, square the differences (so negative and positive differences both
count), add them up and take the square root. Two genes are close only if their expression values are close **in
absolute terms** at every time point.

**Pearson correlation coefficient** (slide 17):

$$s(\mathbf x, \mathbf y) = r = \frac{\frac{1}{J}\sum_{j=1}^{J}(x_j-\mu_x)(y_j-\mu_y)}{\sigma_x\,\sigma_y}\in[-1, 1],
\qquad d_{\text{corr}}(\mathbf x, \mathbf y) = 1 - r .$$

*In words:* centre each profile on its own mean, rescale it by its own spread, and ask whether the two go up and down
**together** ($r = 1$), independently ($r \approx 0$) or in opposition ($r = -1$). Because each profile is centred and
rescaled, $r$ ignores a gene's baseline level and its amplitude — only the **shape** matters. Turning a similarity into a
distance, $1 - r$ runs from 0 (same shape) to 2 (mirror images).

⚠️ **About the slide's formula.** Slide 17 writes $\sum_i (x_i-\mu_x)(y_i-\mu_y)/(\sigma_x\sigma_y)$ without the factor
$\frac1J$ (they call it $\frac1p$). With $\sigma$ defined as above, the $\frac1J$ is needed for $r$ to lie in $[-1, 1]$;
equivalently, one can drop the $\frac1J$ *and* define $\sigma_x = \sqrt{\sum_j (x_j - \mu_x)^2}$. Either way the result is
the usual Pearson $r$, which is what `np.corrcoef` computes.

In [ ]:
# ── 2.1 Euclidean vs correlation distance on four toy gene profiles (SIMULATED) ──
def euclidean(x: np.ndarray, y: np.ndarray) -> float:
    """Straight-line distance sqrt(sum_j (x_j - y_j)^2)."""
    return float(np.sqrt(np.sum((np.asarray(x, float) - np.asarray(y, float)) ** 2)))


def pearson(x: np.ndarray, y: np.ndarray) -> float:
    """Pearson correlation: mean of the product of the two z-scored profiles."""
    zx = (x - x.mean()) / x.std()
    zy = (y - y.mean()) / y.std()
    return float(np.mean(zx * zy))


t_demo = np.arange(0, 120, 10)                                    # 12 time points (minutes)
wave = np.sin(2 * np.pi * t_demo / 60)
genes_demo = {"A": 8 + wave,                                      # baseline 8, amplitude 1
              "B": 11 + 3 * wave,                                 # same shape, higher and larger
              "C": 8 - wave,                                      # same level as A, opposite shape
              "D": 8 + 0.3 * np.cos(2 * np.pi * t_demo / 25)}     # same level, unrelated shape
names = list(genes_demo)
D_euc = pd.DataFrame([[euclidean(genes_demo[a], genes_demo[b]) for b in names] for a in names], names, names)
D_cor = pd.DataFrame([[1 - pearson(genes_demo[a], genes_demo[b]) for b in names] for a in names], names, names)
D_cor = D_cor.clip(lower=0)                                       # remove tiny negative rounding errors (−1e-16)
print("Euclidean distances:\n", D_euc.round(2), "\n\nCorrelation distances 1 − r:\n", D_cor.round(2))
G = np.array([genes_demo[n] for n in names])
print("\nchecks against SciPy/NumPy:", np.allclose(squareform(pdist(G)), D_euc),
      np.allclose(1 - np.corrcoef(G), D_cor))

fig, axes = plt.subplots(1, 3, figsize=(11, 3.2), gridspec_kw=dict(width_ratios=[1.4, 1, 1]))
for n, col in zip(names, ["tab:blue", "tab:orange", "tab:green", "tab:red"]):
    axes[0].plot(t_demo, genes_demo[n], "o-", ms=3, color=col, label=f"gene {n}")
axes[0].set_xlabel("time (min)"); axes[0].set_ylabel("expression (log2)"); axes[0].legend(ncol=2)
axes[0].set_title("four toy genes (SIMULATED)")
for ax, D, title in [(axes[1], D_euc, "Euclidean distance"), (axes[2], D_cor, "correlation distance 1 − r")]:
    ax.imshow(D.values, cmap="viridis_r")
    ax.set_xticks(range(4)); ax.set_xticklabels(names); ax.set_yticks(range(4)); ax.set_yticklabels(names)
    for (i, j), v in np.ndenumerate(D.values):
        ax.text(j, i, f"{v:.1f}", ha="center", va="center", fontsize=8, color="w" if v > D.values.max() / 2 else "k")
    ax.set_title(title + "\n(dark = similar)")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The two distances tell opposite stories about the same genes:

- **Euclidean** distance calls **A and C** close (their values are at the same level, $d \approx 4.9$) and **A and B** far
  apart ($d \approx 11.5$) — even though A and B rise and fall *in perfect synchrony*, while C does exactly the opposite of A.
- **Correlation** distance calls **A and B identical** ($1 - r = 0$) and A and C maximally different ($1 - r = 2$); the
  unrelated gene D is about 1 from everyone.

Which is right depends on the question. If you want **co-regulated genes** — same transcription factor, same programme —
you care about the *shape* of the response, so correlation (or Euclidean distance on z-scored profiles, below) is the
natural choice; this is what Eisen *et al.* used. If absolute amounts matter — protein copy numbers in single cells,
the coordinates of a cell in a 2-D cytometry plot — Euclidean distance is appropriate. Our own `euclidean` and `pearson`
agree with SciPy and NumPy.

### Correlation is Euclidean distance after z-scoring

There is a neat bridge between the two measures. Z-score each profile, $z_j = (x_j - \mu_x)/\sigma_x$, so that it has
mean 0 and $\sum_j z_j^2 = J$. Expanding the square,

$$\lVert \mathbf z_x - \mathbf z_y\rVert^2 = \sum_j z_{x,j}^2 + \sum_j z_{y,j}^2 - 2\sum_j z_{x,j}z_{y,j} = J + J - 2Jr = 2J(1 - r).$$

*In words:* after z-scoring every gene, the squared Euclidean distance **is** the correlation distance, up to the
constant factor $2J$. So "cluster genes by correlation" and "z-score each gene, then use Euclidean distance (and k-means)"
rank pairs of genes in exactly the same order. We use this trick in Section 4.

In [ ]:
# ── 2.2 Check: squared Euclidean distance of z-scored profiles = 2J(1 − r) ──
Zg = (G - G.mean(axis=1, keepdims=True)) / G.std(axis=1, keepdims=True)    # z-score each gene (row)
J = G.shape[1]
lhs = squareform(pdist(Zg)) ** 2
rhs = 2 * J * (1 - np.corrcoef(G))
print("max |‖z_x − z_y‖² − 2J(1 − r)| =", np.abs(lhs - rhs).max().round(12))

🔍 **What just happened?** The identity holds to machine precision for all 16 pairs.

⚠️ **Pitfalls.** (i) $1 - r$ is a convenient *dissimilarity* but not a true metric — it can violate the triangle
inequality; $\sqrt{2(1-r)}$ (the Euclidean distance of profiles z-scored to unit length) is a metric. (ii) Correlation is
undefined for a flat profile ($\sigma = 0$) and very noisy for nearly flat ones: a gene that never changes has no shape
to compare. Filtering out low-variance genes before clustering (Section 8) avoids clustering noise.

✏️ **Your turn.** Add a gene `"E": 8 + wave + rng.normal(0, 0.5, 12)` to `genes_demo` and re-run 2.1. How large is
$1 - r$ between A and E, and how does it change if you increase the noise to 1.0? Noise pushes correlations towards 0.

### 📑 Slide connection — `006_clustering`, slides 15–18

- **What the slides show.** Similarity is philosophically slippery, so take a pragmatic view (15). Clustering features means comparing columns
    of X (16). Distance measures for two objects with *p* features: Euclidean distance and the correlation coefficient
    (17). A distance is a real number D(O₁, O₂) for every pair of objects (18).
- **What this notebook adds.** Both measures implemented and checked against SciPy/NumPy; a four-gene example where they disagree completely;
    the identity ‖z_x − z_y‖² = 2J(1 − r); the usual properties of a distance (the slide only says "a real number");
    and a note that slide 17's correlation formula needs a 1/p factor (or un-normalised σ's).
- **Exam tip.** A classic exam question: "two genes have the same temporal pattern but different expression levels — which
    distance groups them together?" Answer: correlation (or Euclidean after z-scoring each gene), not raw Euclidean.

---
## 3 · A simulated yeast cell-cycle experiment

> 📍 **Where we are.** We have distances; now we need data with a *known* answer so that we can check our algorithms.
> Real time-course experiments (slide 3 of the mixture-model deck shows a yeast microarray time series with one curve per
> gene and k-means centroids) have no answer key, so we simulate one.

**The simulated experiment (SIMULATED — inspired by yeast cell-cycle studies such as Spellman *et al.* 1998).**
Synchronised yeast cells are sampled every 10 minutes for 110 minutes (12 arrays, roughly two pretend 60-minute cycles).
300 genes follow one of four expression **programmes**:

| Programme | Shape | Genes |
|---|---|---|
| G1/S | peaks at 10 and 70 min | 90 |
| S | peaks 15 min later (25, 85 min) | 80 |
| G2/M | peaks at 40 and 100 min (opposite to G1/S) | 70 |
| induced | not periodic: switches on around 55 min (a stress-response-like ramp) | 60 |

Each gene $g$ has its own **baseline** $b_g$ (between 6 and 12 on a log2 scale — some genes are simply more abundant) and
**amplitude** $a_g$ (between 0.5 and 2), plus measurement noise:

$$x_{g}(t) = b_g + a_g\, f_{\text{programme}(g)}(t) + \varepsilon_{g,t}, \qquad \varepsilon_{g,t}\sim\mathcal N(0, 0.4^2).$$

*In words:* genes in the same programme share the shape $f$ but not the level or the size of the swing — exactly the
situation of genes A and B in Section 2.

In [ ]:
# ── 3.1 Simulate the time course (SIMULATED) ──
PROGRAMMES = ["G1/S", "S", "G2/M", "induced"]


def simulate_time_course(rng: np.random.Generator, n_per=(90, 80, 70, 60), noise: float = 0.4):
    """Yeast-like time course: genes × 12 time points, four programmes, gene-specific baseline and amplitude.

    Returns (X, programme_of_each_gene, time_points). Rows are genes, columns are time points.
    """
    t = np.arange(0, 120, 10)
    shapes = np.array([np.cos(2 * np.pi * (t - 10) / 60),       # G1/S
                       np.cos(2 * np.pi * (t - 25) / 60),       # S (a quarter-cycle later)
                       np.cos(2 * np.pi * (t - 40) / 60),       # G2/M (half a cycle later)
                       np.tanh((t - 55) / 15)])                 # induced, non-periodic
    prog = np.repeat(np.arange(4), n_per)
    baseline = rng.uniform(6, 12, len(prog))
    amplitude = rng.uniform(0.5, 2.0, len(prog))
    X = baseline[:, None] + amplitude[:, None] * shapes[prog] + rng.normal(0, noise, (len(prog), len(t)))
    return X, prog, t


rng = np.random.default_rng(SEED + 3)
X_tc, prog_tc, t_tc = simulate_time_course(rng)
print("time-course matrix (genes × time points):", X_tc.shape, "| genes per programme:", np.bincount(prog_tc))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), gridspec_kw=dict(width_ratios=[1, 1.4]))
im = axes[0].imshow(X_tc, aspect="auto", cmap="magma", extent=[-5, 115, len(X_tc), 0])
axes[0].set_xlabel("time (min)"); axes[0].set_ylabel("gene (sorted by true programme)")
axes[0].set_title("raw expression (SIMULATED)"); plt.colorbar(im, ax=axes[0], label="log2 expression")
for k in range(4):
    for g in np.flatnonzero(prog_tc == k)[:4]:
        axes[1].plot(t_tc, X_tc[g], color=CLASS_COLORS[k], lw=1, label=PROGRAMMES[k] if g == np.flatnonzero(prog_tc == k)[0] else None)
axes[1].set_xlabel("time (min)"); axes[1].set_ylabel("log2 expression")
axes[1].set_title("4 genes from each programme: shapes shared, levels not"); axes[1].legend(ncol=4, loc="upper center")
plt.tight_layout(); plt.show()

🔍 **What just happened?** In the heatmap the programmes are hard to see: the strongest pattern is horizontal stripes —
genes with high or low **baselines**. In the right panel you can see the shapes once the curves are coloured by their
(normally unknown!) programme, but the curves of one programme sit at very different heights. A clustering method that
looks at raw levels will group genes by abundance, not by programme. Keep this in mind for Section 4.6.

✏️ **Your turn.** Z-score each gene (subtract its mean over time and divide by its standard deviation, as in cell 2.2) and
redraw the right-hand panel with the z-scored curves. Do the four shapes become easier to tell apart?

---
## 4 · k-means

> 📍 **Where we are.** Slide 19 contrasts two families of clustering methods: **partitional** methods construct a
> partition of the objects into groups and evaluate it with a criterion; **hierarchical** methods build a tree
> (Section 6). k-means is the classic partitional method (slides 20–28).

### 4.1 The objective

Given $N$ samples $\mathbf x_1,\dots,\mathbf x_N$ (each with $J$ features) and a chosen number of clusters $K$, k-means
looks for a partition into clusters $C_1,\dots,C_K$ with centres $\boldsymbol\mu_1,\dots,\boldsymbol\mu_K$ that minimises
the objective shown on slides 30–32:

$$\text{Obj} = \sum_{k=1}^{K}\ \sum_{i \in C_k} \lVert \mathbf x_i - \boldsymbol\mu_k\rVert^2 .$$

*In words:* add up, over all samples, the squared Euclidean distance from each sample to the centre of its own cluster.
Small Obj = tight clusters. (It is also called the *within-cluster sum of squares*, or *inertia* in scikit-learn.)

🧮 **Math refresher — set notation.** $C_k$ is the **set** of sample indices in cluster $k$; "$i \in C_k$" reads
"$i$ is a member of $C_k$"; $|C_k|$ is the **number** of members. $\lVert\mathbf v\rVert^2 = \sum_j v_j^2$ is the squared
length of a vector, so $\lVert \mathbf x_i - \boldsymbol\mu_k\rVert^2$ is the squared Euclidean distance of Section 2.

Slide 20 notes that finding the *globally* best partition would mean enumerating all partitions — astronomically many
(already more than $10^{23}$ ways to split just 50 objects into 3 non-empty groups) — so we use an effective **heuristic**: the
k-means algorithm.

### 4.2 The algorithm (slide 28)

1. Decide on a value for $K$.
2. Initialise the $K$ cluster centres (randomly, if necessary).
3. **Assignment step** — assign each of the $N$ objects to its *nearest* centre.
4. **Update step** — re-estimate each centre as the mean of its members, assuming the memberships are correct:
   $$\boldsymbol\mu_k = \frac{1}{|C_k|}\sum_{i\in C_k}\mathbf x_i .$$
   *In words:* move each centre to the centre of gravity of the samples currently assigned to it.
5. If no object changed membership, stop; otherwise go to 3.

In [ ]:
# ── 4.2 k-means from scratch ──
def kmeans_objective(X: np.ndarray, labels: np.ndarray, centres: np.ndarray) -> float:
    """Sum over samples of the squared distance to the centre of their cluster (slides 30–32)."""
    return float(((X - centres[labels]) ** 2).sum())


def kmeans(X, K: int, init=None, rng=None, max_iter: int = 100):
    """k-means (Lloyd's algorithm). X is N × J; init is an optional K × J array of starting centres.

    Returns (labels, centres, history, snapshots): the objective after every assignment AND every update step,
    and a list of (labels, centres) after each half-step so that we can draw the algorithm at work.
    """
    X = np.asarray(X, float)
    rng = rng if rng is not None else np.random.default_rng(0)
    centres = X[rng.choice(len(X), K, replace=False)].copy() if init is None else np.array(init, float)
    labels, history, snaps = None, [], [(None, centres.copy())]
    for _ in range(max_iter):
        d2 = ((X[:, None, :] - centres[None, :, :]) ** 2).sum(axis=2)    # N × K squared distances
        new_labels = d2.argmin(axis=1)                                   # step 3: nearest centre
        if labels is not None and np.array_equal(new_labels, labels):
            break                                                       # step 5: nobody moved → done
        labels = new_labels
        history.append(kmeans_objective(X, labels, centres)); snaps.append((labels.copy(), centres.copy()))
        for k in range(K):                                               # step 4: centre = mean of members
            if np.any(labels == k):                                      # (an empty cluster keeps its old centre)
                centres[k] = X[labels == k].mean(axis=0)
        history.append(kmeans_objective(X, labels, centres)); snaps.append((labels.copy(), centres.copy()))
    return labels, centres, np.array(history), snaps


# run it on the 60 toy points of Section 1, starting from deliberately poor centres (as on slide 23)
init_bad = np.array([[0.5, 4.5], [1.0, 3.0], [0.8, 0.3]])   # all three on the left
lab_km, cen_km, hist_km, snaps_km = kmeans(X_toy, 3, init=init_bad)
print("objective after each half-step:", np.round(hist_km, 2))
print("never increased:", bool(np.all(np.diff(hist_km) <= 1e-12)), "| final centres:\n", cen_km.round(2))

In [ ]:
# ── 4.3 Watch k-means work (slides 23–27) ──
show = [0, 1, 2, 3, 4, len(snaps_km) - 1]
fig, axes = plt.subplots(2, 3, figsize=(10, 5.8), sharex=True, sharey=True)
for ax, s in zip(axes.ravel(), show):
    labels_s, centres_s = snaps_km[s]
    colours = "0.6" if labels_s is None else [CLASS_COLORS[l] for l in labels_s]
    ax.scatter(*X_toy.T, c=colours, s=12)
    ax.scatter(*centres_s.T, marker="X", s=110, c="k")
    for k, (cx, cy) in enumerate(centres_s):
        ax.annotate(f"μ{k + 1}", (cx, cy), xytext=(4, 4), textcoords="offset points", fontsize=8)
    step = "start" if s == 0 else ("assign" if s % 2 == 1 else "update")
    obj = "" if s == 0 else f"\nObj = {hist_km[s - 1]:.1f}"
    ax.set_title(f"half-step {s}: {step}{obj}", fontsize=9)
for ax in axes[1]:
    ax.set_xlabel("expression in condition 1")
for ax in axes[:, 0]:
    ax.set_ylabel("expression in condition 2")
fig.suptitle("k-means on 60 SIMULATED samples: assign → update → assign … (last panel: converged)", fontsize=10)
plt.tight_layout(); plt.show()

🔍 **What just happened?** From poor starting centres (all three on the left-hand side), the first **assignment** colours
each point by its nearest centre; the **update** moves each centre ($\times$) to the mean of its points; points near the
moving boundaries switch cluster; and after four rounds nothing changes. The printed objective went down at **every** half-step (from 437.4 to 25.6) —
never up. That is not luck.

### 4.4 Why k-means always converges (but not always to the best answer)

**The assignment step cannot increase Obj.** Each sample is moved to the centre that is closest to it, so its term
$\lVert\mathbf x_i-\boldsymbol\mu_{k}\rVert^2$ can only shrink or stay the same.

**The update step cannot increase Obj.** With the memberships fixed, cluster $k$ contributes
$f(\boldsymbol\mu) = \sum_{i\in C_k}\lVert\mathbf x_i - \boldsymbol\mu\rVert^2$. Look at one coordinate $j$ and use the
calculus of [Module 04](04_calculus_and_optimization.ipynb): set the derivative to zero,

$$\frac{\partial f}{\partial \mu_j} = \sum_{i\in C_k} -2\,(x_{ij}-\mu_j) = 0
\;\;\Longrightarrow\;\; \mu_j = \frac{1}{|C_k|}\sum_{i\in C_k}x_{ij}.$$

*In words:* the point that minimises the total squared distance to a group of points is their **mean** (the second
derivative, $2|C_k| > 0$, confirms a minimum). So moving each centre to its cluster mean is the best possible move.

**Therefore** Obj never increases. There are only finitely many ways to assign $N$ points to $K$ clusters, and the
algorithm cannot revisit a partition once Obj has strictly decreased, so it **must stop** after finitely many steps.
But it stops at a **local** optimum — a partition that no single assign-or-update step can improve — which need not be
the global optimum. The answer depends on the starting centres (slide 57: "unsolved issues: … initialization").

⚠️ The objective uses *squared Euclidean* distance, and the mean is the minimiser only for that distance. k-means
therefore prefers compact, roughly round clusters of similar spread, and is sensitive to the scale of each feature —
assumptions that [Module 13](13_mixture_models_em.ipynb) makes explicit with Gaussian mixtures.

### 4.5 Local optima and restarts

The standard remedy is to run k-means from many random starts and keep the run with the **lowest** objective.
A popular smarter start, **k-means++** (Arthur & Vassilvitskii 2007; scikit-learn's default), picks the first centre at
random and each further centre at random with probability proportional to its squared distance from the nearest centre
chosen so far — so the initial centres tend to be spread out.

In [ ]:
# ── 4.5 Many random starts on five groups (SIMULATED) ──
def kmeanspp_init(X: np.ndarray, K: int, rng: np.random.Generator) -> np.ndarray:
    """k-means++ seeding: each new centre is a sample drawn with probability ∝ squared distance to the nearest centre."""
    centres = [X[rng.integers(len(X))]]
    for _ in range(K - 1):
        d2 = ((X[:, None, :] - np.array(centres)[None]) ** 2).sum(axis=2).min(axis=1)
        centres.append(X[rng.choice(len(X), p=d2 / d2.sum())])
    return np.array(centres)


rng = np.random.default_rng(SEED + 45)
centres5 = np.array([[0, 0], [0, 3.2], [3.2, 0], [3.2, 3.2], [7, 1.6]])
X5 = np.vstack([rng.normal(c, 0.55, (40, 2)) for c in centres5])
runs_random = [kmeans(X5, 5, rng=np.random.default_rng(1000 + s)) for s in range(100)]
runs_pp = [kmeans(X5, 5, init=kmeanspp_init(X5, 5, np.random.default_rng(2000 + s))) for s in range(100)]
obj_random = np.array([r[2][-1] for r in runs_random])
obj_pp = np.array([r[2][-1] for r in runs_pp])
best = min(obj_random.min(), obj_pp.min())
print(f"random starts : best Obj {obj_random.min():.1f}, worst {obj_random.max():.1f}, "
      f"{np.mean(obj_random < best + 1e-6):.0%} of 100 runs reach the best, {len(np.unique(obj_random.round(4)))} distinct end points")
print(f"k-means++     : worst {obj_pp.max():.1f}, {np.mean(obj_pp < best + 1e-6):.0%} of 100 runs reach the best")
sk = KMeans(5, n_init=10, random_state=SEED).fit(X5)
print(f"scikit-learn KMeans (10 k-means++ starts): Obj = {sk.inertia_:.1f}")

fig, axes = plt.subplots(1, 3, figsize=(11, 3.2))
axes[0].hist([obj_random, obj_pp], bins=25, label=["random starts", "k-means++ starts"], color=["tab:blue", "tab:orange"])
axes[0].set_xlabel("final objective"); axes[0].set_ylabel("number of runs (of 100)"); axes[0].legend()
axes[0].set_title("where 100 runs end up")
for ax, run, title in [(axes[1], runs_random[int(np.argmax(obj_random))], "worst run"),
                       (axes[2], runs_random[int(np.argmin(obj_random))], "best run")]:
    ax.scatter(*X5.T, c=[CLASS_COLORS[l] for l in run[0]], s=10)
    ax.scatter(*run[1].T, marker="X", s=90, c="k")
    ax.set_title(f"{title}: Obj = {run[2][-1]:.1f} (SIMULATED)"); ax.set_xlabel("feature 1")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The same algorithm on the same data ended in 31 different places. Only half of the 100
random starts found the best partition (objective 115.2); in the worst run (512.6, more than four times larger) two
centres share one group while one centre sits between two groups and has to cover both. k-means++ starts reached the best
partition more often (73 % of runs) but could still get stuck almost as badly — better seeding is not a cure-all. The practical rule: **always use several restarts and keep the lowest objective** (scikit-learn's `n_init`),
and our best run matches scikit-learn's answer.

### 4.6 k-means on the simulated cell-cycle genes

Now we cluster the 300 genes of Section 3 into $K = 4$ groups, once on the **raw** profiles and once after
**z-scoring every gene** (Section 2: Euclidean distance on z-scored profiles = correlation distance). To score a
clustering against the true programmes we use the **adjusted Rand index (ARI)**: it looks at every pair of genes and
asks whether the two partitions agree on "same group" vs "different groups", corrected for chance — ARI = 1 means
identical partitions (up to renaming the clusters), ARI ≈ 0 means no better than random. ([Module 13](13_mixture_models_em.ipynb)
derives it; here we use scikit-learn's `adjusted_rand_score`.)

In [ ]:
# ── 4.6 Raw vs z-scored gene profiles ──
def best_of(X, K, n_starts=10, seed=0):
    """Run k-means from n_starts random starts and keep the run with the lowest objective."""
    runs = [kmeans(X, K, rng=np.random.default_rng(seed + s)) for s in range(n_starts)]
    return min(runs, key=lambda r: r[2][-1])


Z_tc = (X_tc - X_tc.mean(axis=1, keepdims=True)) / X_tc.std(axis=1, keepdims=True)   # z-score each gene
lab_raw = best_of(X_tc, 4)[0]
lab_z, cen_z, _, _ = best_of(Z_tc, 4)
print(f"ARI vs true programmes — raw profiles: {adjusted_rand_score(prog_tc, lab_raw):.2f}   "
      f"z-scored profiles: {adjusted_rand_score(prog_tc, lab_z):.2f}")
print("\nz-scored clusters (rows) × true programmes (columns):")
print(pd.crosstab(lab_z, np.array(PROGRAMMES)[prog_tc], rownames=["cluster"], colnames=["programme"]))
print("\nmean baseline (log2) of the genes in each RAW cluster:",
      np.round([X_tc[lab_raw == k].mean() for k in range(4)], 1))

fig, axes = plt.subplots(1, 4, figsize=(12, 2.6), sharey=True)
for k, ax in enumerate(axes):
    ax.plot(t_tc, Z_tc[lab_z == k].T, color="0.75", lw=0.5)
    ax.plot(t_tc, cen_z[k], color="tab:red", lw=2.5)
    ax.set_title(f"cluster {k}: {np.sum(lab_z == k)} genes"); ax.set_xlabel("time (min)")
axes[0].set_ylabel("z-scored expression")
fig.suptitle("k-means centroids (red) of z-scored SIMULATED gene profiles", fontsize=10)
plt.tight_layout(); plt.show()

🔍 **What just happened?** On the **raw** profiles k-means is essentially useless (ARI near 0): its four clusters are
"low", "medium-low", "medium-high" and "high" baselines — the printed mean levels of the four raw clusters are about
7, 9, 10 and 11 log2 units (in the order the clusters happen to be numbered) — because the differences in level (up
to 6 log2 units) dwarf the cell-cycle swings. After **z-scoring each gene**, the four
clusters match the four programmes almost perfectly (see the contingency table), and the red centroids are clean
versions of the four shapes — compare them with the "K-means centroids" panel of slide 3 of the mixture-model deck.
The algorithm did not change; the **representation and distance** did.

✏️ **Your turn.** (a) Re-simulate with `noise=1.0` in `simulate_time_course` and re-run 4.6. Which programmes get
confused first, and why (hint: compare the shapes of S with its neighbours)? (b) Ask for `K = 6` on the z-scored data:
what happens to the extra clusters?

### 📑 Slide connection — `006_clustering`, slides 19–28

- **What the slides show.** Partitional vs hierarchical clustering (19). k-means: given K, find the partition that optimises a criterion;
    exhaustive search is globally optimal but infeasible, so use heuristics (20). On "expression in condition 1/2" data,
    re-assign and move centres until no object changes membership (21–27, step by step). The algorithm with N samples of p
    features: choose K, initialise, assign to the nearest centroid, set μ_k = (1/|C_k|) Σ_{i∈C_k} x_i, stop when nothing
    changes (28).
- **What this notebook adds.** k-means written from scratch and drawn half-step by half-step; the two-line proof that neither step can increase
    the objective (the mean minimises squared distance — a derivative set to zero); why that guarantees termination but only
    at a local optimum; 100 restarts showing how often k-means gets stuck; k-means++; and the crucial role of z-scoring when
    clustering genes by shape.
- **Exam tip.** Be able to run two k-means iterations by hand on 5–6 points in 2-D, and to explain why the objective cannot
    increase. A common mistake: claiming k-means finds the global optimum — it only finds a local one.

---
## 5 · How many clusters? The elbow

> 📍 **Where we are.** k-means needs $K$ as an input (step 1 of slide 28). Slide 29 is blunt: "How can we tell the
> *right* number of clusters? In general, this is an unsolved problem." Slides 30–33 show one approximate method.

The slides run k-means with $K = 1, 2, 3, \dots$ on a two-group toy and record the objective: **873.0** for $K = 1$,
**173.1** for $K = 2$ and **133.6** for $K = 3$. Plotted against $K$ (slide 33), the curve drops abruptly at $K = 2$ and
then flattens: an **elbow**, "highly suggestive of two clusters in the data".

Why does the curve always go down? With more centres every sample can be at least as close to some centre as before —
the best possible objective for $K+1$ clusters is never worse than for $K$ (at $K = N$ it is 0: every sample its own
centre). So we cannot simply pick the $K$ with the lowest objective; we look for the point where adding clusters stops
paying off.

In [ ]:
# ── 5.1 The slides' two-group toy, rebuilt (SIMULATED) ──
rng = np.random.default_rng(SEED + 51)
X_two = np.vstack([rng.normal([2.5, 3.8], [0.9, 1.3], (25, 2)),        # lower-left group
                   rng.normal([8.0, 8.0], [0.9, 0.9], (25, 2))])       # upper-right group
Ks = range(1, 7)
obj_two = [best_of(X_two, K, n_starts=10)[2][-1] for K in Ks]
print("objective for K = 1..6:", np.round(obj_two, 1))
print(f"drop from K=1 to 2: {obj_two[0] - obj_two[1]:.1f};  from K=2 to 3: {obj_two[1] - obj_two[2]:.1f}")

fig, axes = plt.subplots(1, 2, figsize=(9, 3.4))
lab2, cen2, _, _ = best_of(X_two, 2)
axes[0].scatter(*X_two.T, c=[CLASS_COLORS[l] for l in lab2], s=18)
axes[0].scatter(*cen2.T, marker="X", s=120, c="k")
axes[0].set_xlim(0, 10.5); axes[0].set_ylim(0, 10.5); axes[0].set_aspect("equal")
axes[0].set_title(f"K = 2, Obj = {obj_two[1]:.1f} (SIMULATED)"); axes[0].set_xlabel("feature 1"); axes[0].set_ylabel("feature 2")
axes[1].plot(list(Ks), obj_two, "o-", color="tab:red")
axes[1].set_xlabel("number of clusters K"); axes[1].set_ylabel("objective (within-cluster SS)")
axes[1].set_title("the elbow at K = 2 (compare slide 33)")
plt.tight_layout(); plt.show()

🔍 **What just happened?** Our toy is not the slides' exact data set (theirs is not provided), so the numbers differ —
711.8 → 119.7 → 76.7 here vs 873.0 → 173.1 → 133.6 on the slides — but the shape is the same: a huge drop from $K = 1$ to
$K = 2$ and only small gains afterwards, a clear elbow. Each
extra centre beyond 2 merely splits a genuine group in two and reduces the objective a little.

The slide adds a warning in small print: "the results are not always as clear cut as in this toy example". Let us look
at a case with a clear answer and a real case.

In [ ]:
# ── 5.2 Elbows on the simulated time course and on real leukaemia samples ──
var_order = np.argsort(X_golub.var(axis=0))[::-1]                 # genes from most to least variable
Xg500 = X_golub[:, var_order[:500]]
Xg500 = Xg500 - Xg500.mean(axis=0)                                # centre each gene
Ks = range(1, 11)
elbow_tc = np.array([best_of(Z_tc, K)[2][-1] for K in Ks])
elbow_golub = np.array([best_of(Xg500, K)[2][-1] for K in Ks])

fig, axes = plt.subplots(1, 2, figsize=(10, 3.3))
for ax, curve, title in [(axes[0], elbow_tc, "SIMULATED time course (4 true programmes)"),
                         (axes[1], elbow_golub, f"Golub patients, 500 most variable genes\n({golub['source']})")]:
    ax.plot(list(Ks), curve / curve[0], "o-")
    ax.set_xlabel("number of clusters K"); ax.set_ylabel("objective / objective at K = 1"); ax.set_title(title)
plt.tight_layout(); plt.show()
print("time course, relative objective:", np.round(elbow_tc / elbow_tc[0], 2))
print("Golub,       relative objective:", np.round(elbow_golub / elbow_golub[0], 2))

🔍 **What just happened?** For the simulated time course the curve bends sharply at $K = 4$ — the true number of
programmes (relative objective 1 → 0.60 → 0.37 → 0.21, then flat). For the 72 real leukaemia patients there is **no
clear elbow**: after a 17 % drop from $K = 1$ to $K = 2$ the objective keeps falling by about 1–8 % of its $K = 1$
value with every extra cluster (0.83 → 0.76 → 0.72 → … → 0.58), and
different people could argue for 2, 3 or 5. That is typical of real biological data:
groups overlap, there are subgroups within groups (ALL itself has B-cell and T-cell forms), and with only 72 samples in
500 dimensions every extra centre fits some noise.

⚠️ The elbow is a heuristic, not a test. Better-founded criteria exist — the silhouette and probabilistic criteria such
as BIC or held-out likelihood ([Module 13](13_mixture_models_em.ipynb)) and cross-validation-style stability checks
([Module 23](23_model_selection.ipynb)) — but none of them can tell you the "true" number of biological types, because
that depends on the resolution you care about.

✏️ **Your turn.** In 5.1 move the second group closer, e.g. to `[5.0, 5.5]`, and re-run. How large is the drop from
$K = 1$ to $K = 2$ now, and is the elbow still convincing?

### 📑 Slide connection — `006_clustering`, slides 29–33

- **What the slides show.** Choosing the right number of clusters is in general unsolved (29). On a two-group toy the objective
    Σ_k Σ_{i∈C_k} ‖x_i − μ_k‖² is 873.0 (K = 1), 173.1 (K = 2) and 133.6 (K = 3) (30–32); the plot for K = 1…6 drops abruptly
    at K = 2, "highly suggestive of two clusters", with the caveat that real results are rarely this clear (33).
- **What this notebook adds.** Why the optimal objective can only decrease with K; a rebuilt toy with the same elbow (the slides' raw points are
    not available, so our numbers differ); and two contrasting cases — a clean elbow at the true K = 4 on simulated genes
    and no elbow at all on the real Golub patients.
- **Exam tip.** Know why you cannot choose K by minimising the objective (it reaches 0 at K = N) and be able to sketch an
    elbow plot and say what an "abrupt change" suggests — and what it does not prove.

---
## 6 · Hierarchical (agglomerative) clustering

> 📍 **Where we are.** k-means returns one flat partition for one chosen $K$. **Hierarchical clustering** (slides 34–52)
> builds a whole **tree** of nested groupings — from every object on its own to everything in one group — and lets you
> decide afterwards where to cut it.

**Bottom-up (agglomerative) recipe** (slide 35): *start with each item in its own cluster, find the best pair to merge
into a new cluster, repeat until all clusters are fused together.* Slides 37–40 animate it: consider all possible
merges, choose the best, repeat. The result is drawn as a **dendrogram** — a tree whose branch heights record how far
apart the two merged groups were.

It all starts from a **distance matrix** (slide 36) containing the distances between every pair of objects. The slide's
five objects are cartoon characters — from left to right Edna, Lisa, Marge, Selma and Patty — with
$D(\text{Edna}, \text{Lisa}) = 8$ and $D(\text{Selma}, \text{Patty}) = 1$ (the twins). As soon as a cluster contains
more than one object we need the question of slide 40: **how do we compute distances between clusters rather than
objects?** Three answers (slides 41–44), for clusters $A$ and $B$:

$$d_{\text{single}}(A,B)=\min_{a\in A,\ b\in B} d(a,b),\qquad
d_{\text{complete}}(A,B)=\max_{a\in A,\ b\in B} d(a,b),\qquad
d_{\text{average}}(A,B)=\frac{1}{|A|\,|B|}\sum_{a\in A}\sum_{b\in B} d(a,b).$$

*In words:* **single link** = distance between the two *closest* members (can produce "long and skinny" clusters);
**complete link** = distance between the two *farthest* members (gives tight clusters); **average link** = average over
*all* cross-pairs ("the most widely used measure", "robust against noise").

In [ ]:
# ── 6.1 Single-link merges by hand, printed step by step ──
def label_of(members) -> str:
    """Name a cluster like the slides: (1,2,3)."""
    return "(" + ",".join(str(m) for m in members) + ")" if len(members) > 1 else str(members[0])


def merge_by_hand(D, names, linkage: str = "single") -> None:
    """Print every merge of agglomerative clustering, with the cluster-distance formulas used."""
    clusters = [[n] for n in names]
    idx = {n: i for i, n in enumerate(names)}
    D = np.asarray(D, float)
    rule = {"single": ("min", np.min), "complete": ("max", np.max), "average": ("mean", np.mean)}[linkage]
    def cdist(A, B):
        return rule[1]([D[idx[a], idx[b]] for a in A for b in B])
    while len(clusters) > 1:
        pairs = [(cdist(A, B), i, j) for i, A in enumerate(clusters) for j, B in enumerate(clusters) if i < j]
        d, i, j = min(pairs)
        merged = clusters[i] + clusters[j]
        print(f"merge {label_of(clusters[i])} + {label_of(clusters[j])} at height {d:g}")
        rest = [C for k, C in enumerate(clusters) if k not in (i, j)]
        A, B, M = label_of(clusters[i]), label_of(clusters[j]), label_of(merged)
        for C in rest:
            c, new = label_of(C), cdist(merged, C)
            if linkage == "average":
                print(f"    d_{M},{c} = mean of {len(merged) * len(C)} pairwise distances = {new:g}")
            else:
                p0, p1 = cdist(clusters[i], C), cdist(clusters[j], C)
                print(f"    d_{M},{c} = {rule[0]}{{d_{A},{c}, d_{B},{c}}} = {rule[0]}{{{p0:g}, {p1:g}}} = {new:g}")
        clusters = [merged] + rest


# Slide 36: Edna, Lisa, Marge, Selma, Patty (upper triangle as on the slide, mirrored here)
D36 = np.array([[0, 8, 8, 7, 7],
                [8, 0, 2, 4, 4],
                [8, 2, 0, 3, 3],
                [7, 4, 3, 0, 1],
                [7, 4, 3, 1, 0]])
print("Slide 36, single link:")
merge_by_hand(D36, ["Edna", "Lisa", "Marge", "Selma", "Patty"], "single")

🔍 **What just happened?** The twins Selma and Patty merge first (distance 1), then Lisa and Marge (2). The distance
between the two pairs is the smallest of the four cross-distances, $\min\{4, 4, 3, 3\} = 3$, so they merge at height 3,
and Edna — far from everyone — joins last at $\min\{8, 8, 7, 7\} = 7$. This is the tree drawn on slides 35 and 40.

### The worked example of slides 45–51

The "Example: single link" slides use a **different** $5 \times 5$ matrix with objects numbered 1–5:

$$D=\begin{pmatrix}0&2&6&10&9\\2&0&3&9&8\\6&3&0&7&5\\10&9&7&0&4\\9&8&5&4&0\end{pmatrix}.$$

*In words:* entry $(i, j)$ is the distance between objects $i$ and $j$; the matrix is symmetric with zeros on the
diagonal, so the slides only print its lower triangle.

Objects 1 and 2 are closest ($d = 2$) and merge first; the slide then updates the matrix with
$d_{(1,2),3}=\min\{d_{1,3}, d_{2,3}\}=\min\{6,3\}=3$, $d_{(1,2),4}=\min\{10,9\}=9$, $d_{(1,2),5}=\min\{9,8\}=8$, and so on.
Let the code reproduce every line.

In [ ]:
# ── 6.2 The slides' worked example (slides 45–51) ──
D45 = np.array([[0, 2, 6, 10, 9],
                [2, 0, 3, 9, 8],
                [6, 3, 0, 7, 5],
                [10, 9, 7, 0, 4],
                [9, 8, 5, 4, 0]])
print("Slides 45–51, single link:")
merge_by_hand(D45, [1, 2, 3, 4, 5], "single")

🔍 **What just happened?** Every number matches the slides: merge (1,2) at 2; then $d_{(1,2),3}=3$, $d_{(1,2),4}=9$,
$d_{(1,2),5}=8$ (slides 47–49); merge (1,2,3) at 3, with $d_{(1,2,3),4}=\min\{9,7\}=7$ and $d_{(1,2,3),5}=\min\{8,5\}=5$
(slide 50); merge (4,5) at 4; finally $d_{(1,2,3),(4,5)}=\min\{7,5\}=5$ (slide 51). The four merge heights 2, 3, 4, 5 are
the heights of the dendrogram's horizontal bars.

✏️ **Your turn (by hand first!).** Repeat the slides 45–51 example with **complete** and **average** linkage on paper.
Then check yourself with `merge_by_hand(D45, [1, 2, 3, 4, 5], "complete")` and `"average"`. (Answers: complete link merges
at 2, 4, 6, 10; average link at 2, 4, 4.5, 8 — same tree shape, different heights.)

### 6.3 A general implementation

Recomputing every cluster distance from scratch is wasteful. All three linkages have a simple **update rule**: when
clusters $A$ and $B$ merge into $A\cup B$, its distance to any other cluster $C$ is

$$d(A\cup B, C) = \min\{d(A,C), d(B,C)\}\ \text{(single)},\quad \max\{\cdot,\cdot\}\ \text{(complete)},\quad
\frac{|A|\,d(A,C)+|B|\,d(B,C)}{|A|+|B|}\ \text{(average)}.$$

*In words:* the new row of the distance matrix is computed from the two old rows — exactly what the slides do when they
write $d_{(1,2),3}=\min\{d_{1,3},d_{2,3}\}$. (This family of updates is known as the Lance–Williams formula.) Our
function returns the merges in SciPy's "linkage matrix" format, one row per merge: *[cluster a, cluster b, height,
size]*, where original objects are numbered $0,\dots,n-1$ and the cluster formed at merge $m$ gets number $n+m$.
That lets SciPy draw our trees.

In [ ]:
# ── 6.3 Agglomerative clustering from scratch (single / complete / average) ──
def agglomerative(D, linkage: str = "average") -> np.ndarray:
    """Naive O(n³) agglomerative clustering on a full distance matrix D (n × n).

    Returns an (n−1) × 4 linkage matrix [id_a, id_b, height, size], the format used by scipy.cluster.hierarchy.
    """
    dist = np.array(D, float)
    n = len(dist)
    np.fill_diagonal(dist, np.inf)
    size, ids, active = np.ones(n), np.arange(n), np.ones(n, bool)
    Z = []
    for m in range(n - 1):
        masked = np.where(active[:, None] & active[None, :], dist, np.inf)
        a, b = np.unravel_index(np.argmin(masked), masked.shape)      # closest pair of active clusters
        a, b = min(a, b), max(a, b)
        Z.append([min(ids[a], ids[b]), max(ids[a], ids[b]), dist[a, b], size[a] + size[b]])
        if linkage == "single":
            new = np.minimum(dist[a], dist[b])
        elif linkage == "complete":
            new = np.maximum(dist[a], dist[b])
        else:                                                           # average: size-weighted mean
            new = (size[a] * dist[a] + size[b] * dist[b]) / (size[a] + size[b])
        dist[a, :] = dist[:, a] = new                                   # the merged cluster lives in row a
        dist[a, a] = np.inf
        active[b] = False
        size[a] += size[b]
        ids[a] = n + m
    return np.array(Z)


fig, axes = plt.subplots(1, 3, figsize=(11, 3), sharey=True)
for ax, method in zip(axes, ["single", "complete", "average"]):
    Z_ours = agglomerative(D45, method)
    Z_scipy = hierarchy.linkage(squareform(D45), method=method)
    same = np.allclose(np.sort(Z_ours[:, 2]), np.sort(Z_scipy[:, 2]))
    print(f"{method:8s}: merge heights {Z_ours[:, 2]}   identical to SciPy: {same}")
    hierarchy.dendrogram(Z_ours, labels=["1", "2", "3", "4", "5"], ax=ax, color_threshold=0, above_threshold_color="k")
    ax.set_title(f"{method} link (slides 45–51 matrix)"); ax.set_xlabel("object")
axes[0].set_ylabel("merge height = cluster distance")
plt.tight_layout(); plt.show()

🔍 **What just happened?** Our 25-line function reproduces SciPy's merge heights for all three linkages, and the
single-link dendrogram is the one built up on slides 45–51: bars at heights 2, 3, 4, 5. On this matrix all three linkages
agree on the *shape* of the tree — {1, 2, 3} vs {4, 5} — and differ only in the heights (complete link always reports
the largest, single link the smallest). On realistic data they can disagree on the shape too.

### 6.4 Single-link chaining vs complete-link compactness

Slide 42 warns that single link produces "potentially long and skinny clusters", and slide 43 credits complete link
with "tight clusters". The reason: single link merges two clusters as soon as **any one pair** of members is close, so a
thin trail of intermediate points can chain two distant groups together. Below, two cell populations (A and B) are
connected by a trail of 12 intermediate cells — think of cells caught in a differentiation path — and a third
population C sits apart. We cut each tree into three clusters with `hierarchy.fcluster`.

In [ ]:
# ── 6.4 Chaining (SIMULATED) ──
rng = np.random.default_rng(SEED + 64)
A_ = rng.normal([0, 0], 0.4, (30, 2)); B_ = rng.normal([5, 0], 0.4, (30, 2)); C_ = rng.normal([2.5, 3.2], 0.4, (30, 2))
trail = np.c_[np.linspace(0.9, 4.1, 12), rng.normal(0, 0.08, 12)]
X_chain = np.vstack([A_, B_, C_, trail])
D_chain = squareform(pdist(X_chain))

fig, axes = plt.subplots(1, 3, figsize=(11, 3.2), sharex=True, sharey=True)
for ax, method in zip(axes, ["single", "complete", "average"]):
    cut = hierarchy.fcluster(agglomerative(D_chain, method), 3, criterion="maxclust")
    ax.scatter(*X_chain.T, c=[CLASS_COLORS[c - 1] for c in cut], s=14)
    ax.set_title(f"{method} link, cut into 3 clusters\nsizes {sorted(np.bincount(cut)[1:].tolist(), reverse=True)}")
    ax.set_xlabel("feature 1")
axes[0].set_ylabel("feature 2")
fig.suptitle("two populations joined by a trail of intermediate cells, plus a third population (SIMULATED)", fontsize=10)
plt.tight_layout(); plt.show()

🔍 **What just happened?** **Single link** chains A, the trail and B into one long cluster (71 points); to make its
third cluster it splits off a single stray point (just above A) rather than cutting the chain. **Complete** and **average link**
recover the three compact populations and divide the trail between A and B. Neither answer is "wrong": if the trail is a
real continuum of cell states, single link describes the topology honestly; if you want compact cell types, complete or
average link is the better tool. Knowing what each linkage favours is the point of slides 41–44.

⚠️ Single link is also very sensitive to noise: one outlying point between two groups is enough to fuse them. That is
why average link is described as "robust against noise" and is the default in most expression analyses.

In [ ]:
# ── 6.5 Single vs average linkage dendrograms of the two-group toy (slide 52) ──
D_two = squareform(pdist(X_two))
fig, axes = plt.subplots(1, 2, figsize=(11, 3.2), sharey=False)
for ax, method in zip(axes, ["single", "average"]):
    Zt = agglomerative(D_two, method)
    hierarchy.dendrogram(Zt, ax=ax, no_labels=True, color_threshold=Zt[-2, 2] + 1e-9, above_threshold_color="k")
    ax.set_title(f"{method} linkage — top merge at {Zt[-1, 2]:.2f}, next at {Zt[-2, 2]:.2f}")
    ax.set_xlabel("50 SIMULATED samples"); ax.set_ylabel("merge height (Euclidean)")
plt.tight_layout(); plt.show()

🔍 **What just happened?** As on slide 52 (*"height represents distance between objects / clusters"*), both trees
split the 50 points of Section 5 into the same two groups at the top, and in both the final merge stands well above all
others (single: 3.70 vs 1.48 for the next merge; average: 7.08 vs 2.97) — that gap between the last and second-to-last
merge heights is what makes two clusters obvious. Below the top the trees look different. With **single** linkage the
heights are compressed — every merge height is a nearest-neighbour distance, so no other bar exceeds 1.5 — and points
or small groups join a growing cluster one at a time, a staircase (the chaining look). With **average** linkage the
sub-clusters are more balanced and their heights more spread out.

### 📑 Slide connection — `006_clustering`, slides 34–52

- **What the slides show.** Bottom-up agglomerative clustering: start from singletons, repeatedly merge the best pair (35–40), starting from
    a distance matrix of five cartoon characters (36). Cluster distances: single link = closest members (long, skinny
    clusters), complete link = farthest members (tight clusters), average link = all pairs (most widely used, robust to
    noise) (41–44). Worked single-link example on a 5-object matrix with updates such as d_(1,2),3 = min{6,3} = 3 (45–51).
    Single vs average linkage dendrograms; height = distance (52).
- **What this notebook adds.** Both slide matrices worked through by code that prints every update; complete and average link as exercises with
    answers; a from-scratch O(n³) implementation using the update rules, checked against SciPy; a chaining experiment; and
    the observation that slide 36 and slides 45–51 use *different* matrices (easy to mix up when revising).
- **Exam tip.** Expect to perform single-, complete- or average-link merges on a 4–6 object matrix by hand and draw the
    dendrogram with correct heights. Remember: complete link uses the max, average link averages over all |A|·|B| pairs —
    not the average of the two cluster-to-cluster distances unless the clusters have equal size.

---
## 7 · Properties of hierarchical clustering

> 📍 **Where we are.** We can build trees. Slides 53–55 summarise their strengths and weaknesses; we test three of them.

From slide 53:

- ✅ **No need to specify the number of clusters in advance** — you get every $K$ at once, by cutting the tree at
  different heights.
- ✅ **The hierarchy maps nicely onto human intuition** — biologists are used to trees (taxonomies, phylogenies).
- ❌ **It does not scale well:** time complexity of at least $O(n^2)$ — there are $n(n-1)/2$ pairwise distances to compute
  and store; our naive implementation (searching the whole matrix at each of the $n-1$ merges) costs $O(n^3)$.
- ❌ **Like any heuristic search, local optima are a problem:** merges are greedy and never undone. A pair merged early
  stays together even if a better global arrangement would separate them.
- ❌ **Interpretation is (very) subjective:** where to cut, which linkage, which distance.

🧮 **Math refresher — big-O notation.** $O(n^2)$ means "grows roughly like $n^2$": double the number of objects and the
work goes up about $2^2 = 4$ times; for $O(n^3)$, about $2^3 = 8$ times.

In [ ]:
# ── 7.1 How the cost grows ──
rng = np.random.default_rng(SEED + 71)
for n in [100, 200, 400]:
    Dn = squareform(pdist(rng.normal(size=(n, 2))))
    t0 = time.perf_counter(); agglomerative(Dn, "average"); t_ours = time.perf_counter() - t0
    t0 = time.perf_counter(); hierarchy.linkage(squareform(Dn, checks=False), "average"); t_sp = time.perf_counter() - t0
    print(f"n = {n:4d}: {n * (n - 1) // 2:6d} pairwise distances | our naive O(n³) code {t_ours:6.3f} s | SciPy {t_sp:.4f} s")

🔍 **What just happened?** Each doubling of $n$ makes our naive code several times slower (roughly the factor 8
expected for $O(n^3)$, blurred by fixed overheads), while SciPy's optimised $O(n^2)$ algorithms stay fast. Even so, memory
is the real wall: 20,000 genes need $2\times10^8$ pairwise distances (1.6 GB as 8-byte numbers). k-means, by contrast,
costs only $O(NKJ)$ per iteration (slide 78 of the mixture deck calls it "fastest").

### 7.2 Outliers and cutting the tree

Slide 55 points out a nice use of dendrograms: **"the single isolated branch is suggestive of a data point that is very
different to all others"** — an outlier. We add one unusual sample to the two-group toy and look at the tree; we also
cut the tree at several heights to see how the number of clusters depends on the cut.

In [ ]:
# ── 7.2 An outlier shows up as an isolated branch (SIMULATED) ──
X_out = np.vstack([X_two, [[6.0, 0.7]]])                    # one unusual sample, bottom middle (as on slide 55)
Z_out = agglomerative(squareform(pdist(X_out)), "average")
outlier = len(X_out) - 1
row = np.flatnonzero((Z_out[:, 0] == outlier) | (Z_out[:, 1] == outlier))[0]
print(f"the outlier joins at height {Z_out[row, 2]:.2f}; merge {row + 1} of {len(Z_out)} "
      f"(the median merge height is {np.median(Z_out[:, 2]):.2f})")
print(f"top merge (the two big groups) at height {Z_out[-1, 2]:.2f}")
for h in [6.0, 5.0, 2.0, 1.0]:
    k = len(np.unique(hierarchy.fcluster(Z_out, h, criterion="distance")))
    print(f"cut at height {h:3.1f} → {k} clusters")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.2), gridspec_kw=dict(width_ratios=[1, 2]))
axes[0].scatter(*X_out[:-1].T, s=14, color="tab:green"); axes[0].scatter(*X_out[-1], s=60, color="tab:red", marker="D")
axes[0].annotate("outlier", X_out[-1], xytext=(10, 10), textcoords="offset points")
axes[0].set_xlim(0, 10.5); axes[0].set_ylim(0, 10.5); axes[0].set_aspect("equal"); axes[0].set_title("data (SIMULATED)")
hierarchy.dendrogram(Z_out, ax=axes[1], labels=[""] * outlier + ["outlier"], color_threshold=0, above_threshold_color="k")
for h, ls in [(6.0, "--"), (5.0, ":")]:
    axes[1].axhline(h, color="tab:blue", ls=ls, lw=1, label=f"cut at {h}")
axes[1].set_ylabel("average-linkage height"); axes[1].legend(); axes[1].set_title("dendrogram: the outlier is a lone branch")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The outlier does not join any group until merge 49 of 50, at height 5.35 — almost ten times
the median merge height — so it hangs from the top of the tree as a lone branch, just like slide 55. The cuts show how
subjective the "number of clusters" is: cutting at 6 gives two clusters (the outlier hidden inside one of them), cutting
at 5 gives three (the outlier becomes a cluster of its own), and cutting at 2 or 1 gives many small groups. Slide 54 asks "But what are the
clusters?" — the tree does not answer; you do.

⚠️ In real data an isolated branch can be a mislabelled or failed sample, a contaminated culture — or the most
interesting patient in the cohort. Check before you delete.

✏️ **Your turn.** Move the outlier to `[5.0, 5.0]`, half-way between the two groups, and re-run 7.2 with `"average"` and
then `"single"` linkage. Is it still an isolated branch? Which linkage now lets it bridge the two groups?

### 📑 Slide connection — `006_clustering`, slides 53–55

- **What the slides show.** Hierarchical clustering needs no K, maps onto intuition, but scales at least as O(n²), can get stuck in local
    optima, and is subjective to interpret (53). Sometimes the correct number of clusters is clear, but rarely (54). A
    single isolated branch suggests an outlier (55).
- **What this notebook adds.** Timings of a naive O(n³) implementation vs SciPy; the memory cost of all pairwise distances; the greedy, never-undone
    nature of merges; an outlier experiment; and cutting one tree at several heights to show how the number of clusters
    depends on the cut.
- **Exam tip.** Typical short-answer question: "give two advantages and two disadvantages of hierarchical clustering compared
    with k-means." Mention: no K needed / a full tree vs O(n²) memory and time / subjective cut / greedy merges.

---
## 8 · Real data: class discovery in leukaemia

> 📍 **Where we are.** We have two clustering algorithms, written from scratch and checked. Now we use them for what
> Golub *et al.* (1999) called **class discovery**: if nobody had told us which patients had ALL and which had AML, would
> the expression data alone have split them that way?

**The data.** Bone-marrow or peripheral-blood samples from 72 leukaemia patients (47 ALL, 25 AML), measured on
Affymetrix arrays and preprocessed as in Dudoit, Fridlyand & Speed (2002): intensities thresholded to the range
100–16,000, genes with little variation removed (3,571 genes remain) and $\log_{10}$-transformed.

**The analysis plan — fixed *before* looking at the labels.** Deciding the analysis after seeing how well it matches the
labels would be circular, so we commit to a standard pipeline:

1. keep the **500 most variable genes** (a common filter: low-variance genes carry mostly noise — Section 2's pitfall);
2. **centre each gene** (subtract its mean over the 72 patients), so that patients are compared by which genes are
   relatively up or down;
3. **hierarchical clustering** of the patients with **correlation distance** and **average linkage** (our
   `agglomerative`), cut into 2 clusters; and **k-means** with $K = 2$ (best of 20 starts) on the same matrix;
4. only then compare the clusters with the ALL/AML labels.

In [ ]:
# ── 8.1 Prepare the matrix and cluster the 72 patients ──
print("source:", golub["source"])
y_golub = golub["y"]                                             # 0 = ALL, 1 = AML — used ONLY for evaluation
print(f"{len(y_golub)} patients: {np.sum(y_golub == 0)} ALL, {np.sum(y_golub == 1)} AML; using {Xg500.shape[1]} genes")

D_pat = squareform(pdist(Xg500, metric="correlation"))          # 1 − r between patients
Z_pat = agglomerative(D_pat, "average")
Z_pat_scipy = hierarchy.linkage(pdist(Xg500, metric="correlation"), "average")
print("our tree = SciPy's tree (merge heights):", np.allclose(np.sort(Z_pat[:, 2]), np.sort(Z_pat_scipy[:, 2])))
hc2 = hierarchy.fcluster(Z_pat, 2, criterion="maxclust") - 1
km2 = best_of(Xg500, 2, n_starts=20)[0]

labels_txt = np.where(y_golub == 1, "AML", "ALL")
for name, lab in [("hierarchical (average link, correlation)", hc2), ("k-means (K = 2)", km2)]:
    print(f"\n{name}: ARI = {adjusted_rand_score(y_golub, lab):.2f}")
    print(pd.crosstab(lab, labels_txt, rownames=["cluster"], colnames=["diagnosis"]))

🔍 **What just happened?** Without seeing a single label, average-linkage clustering split the patients into two groups
that almost coincide with the diagnoses: one cluster holds 24 of the 25 AML patients, the other all 47 ALL patients
plus one AML patient (ARI = 0.94). k-means misplaces three patients (two ALL, one AML; ARI = 0.84). Our implementation
builds exactly SciPy's tree. The dominant structure in leukaemia expression *is* the lineage of the cancer cell —
lymphoid vs myeloid — which is why Golub *et al.* could rediscover the two classes from expression alone (they used a
different clustering method, self-organising maps).

Let us look at the data the way biologists usually do: a **heatmap with the samples ordered by the tree**.

In [ ]:
# ── 8.2 Heatmap: patients ordered by our tree, genes ordered by their own tree ──
Z_genes = hierarchy.linkage(pdist(Xg500.T, metric="correlation"), "average")     # cluster the 500 genes too
gene_order = hierarchy.leaves_list(Z_genes)
fig = plt.figure(figsize=(10, 6.2))
gs = fig.add_gridspec(3, 2, height_ratios=[1.1, 0.12, 4], width_ratios=[40, 1], hspace=0.04, wspace=0.03)
ax_d, ax_s, ax_h, ax_c = fig.add_subplot(gs[0, 0]), fig.add_subplot(gs[1, 0]), fig.add_subplot(gs[2, 0]), fig.add_subplot(gs[2, 1])
dn = hierarchy.dendrogram(Z_pat, ax=ax_d, no_labels=True, color_threshold=0, above_threshold_color="k")
ax_d.set_ylabel("1 − r"); ax_d.set_title(f"Golub leukaemia: 72 patients × 500 genes, average linkage ({golub['source']})")
ax_d.set_xticks([])
pat_order = dn["leaves"]
ax_s.imshow(y_golub[pat_order][None, :], aspect="auto", cmap=ListedColormap(CLASS_COLORS[:2]))
ax_s.set_yticks([0]); ax_s.set_yticklabels(["ALL / AML"], fontsize=7); ax_s.set_xticks([])
im = ax_h.imshow(Xg500[np.ix_(pat_order, gene_order)].T, aspect="auto", cmap="RdBu_r", vmin=-1, vmax=1)
ax_h.set_xlabel("patients (ordered by the tree; strip above: blue = ALL, red = AML)")
ax_h.set_ylabel("500 genes (ordered by gene tree)"); ax_h.set_yticks([]); ax_h.set_xticks([])
fig.colorbar(im, cax=ax_c, label="log10 expression − gene mean")
plt.show()

gene_clusters = hierarchy.fcluster(Z_genes, 4, criterion="maxclust")
summary = pd.DataFrame({"genes": np.bincount(gene_clusters)[1:],
                        "mean in ALL": [Xg500[y_golub == 0][:, gene_clusters == g].mean() for g in range(1, 5)],
                        "mean in AML": [Xg500[y_golub == 1][:, gene_clusters == g].mean() for g in range(1, 5)]},
                       index=[f"gene cluster {g}" for g in range(1, 5)])
print("centred expression in the four groups of a 4-cluster cut of the gene tree:\n", summary.round(2))

🔍 **What just happened?** Clustering **both** axes turns a 72 × 500 wall of numbers into blocks (compare slides 6, 7
and 56). The colour strip shows the patient tree separating red (AML) from blue (ALL). The gene tree groups genes that
rise and fall together across patients. Cutting it into four groups gives two large blocks — 177 genes that are on
average **higher in ALL** and 283 genes that are **higher in AML** (positive centred expression = above that gene's
average) — and two small groups with no ALL/AML difference, which follow other patterns visible in the heatmap. These
co-expressed blocks are candidates for lineage-specific programmes — exactly the "guilt by association" reasoning of the
biological question.

### 8.3 How much did our choices matter?

Our pipeline made four choices: the number of genes, centring, the distance and the linkage. A responsible analysis asks
how stable the conclusion is when those choices change. Below we repeat the analysis with 100–2,000 genes and every
linkage (SciPy here, for speed — Section 6 showed it gives the same trees as ours).

In [ ]:
# ── 8.3 Sensitivity of the ALL/AML agreement (ARI) to analysis choices ──
rows = {}
for n_genes in [100, 250, 500, 1000, 2000]:
    Xs = X_golub[:, var_order[:n_genes]]
    Xs = Xs - Xs.mean(axis=0)
    d = pdist(Xs, metric="correlation")
    row = {}
    for method in ["average", "complete", "single"]:
        lab = hierarchy.fcluster(hierarchy.linkage(d, method), 2, criterion="maxclust")
        row[f"{method} link"] = adjusted_rand_score(y_golub, lab)
    row["k-means"] = adjusted_rand_score(y_golub, best_of(Xs, 2, n_starts=20)[0])
    rows[n_genes] = row
table = pd.DataFrame(rows).T
table.index.name = "genes kept"
print("ARI between the 2-cluster solution and ALL/AML:\n", table.round(2))
lab_single = hierarchy.fcluster(hierarchy.linkage(pdist(Xg500, metric="correlation"), "single"), 2, criterion="maxclust")
print("\nsingle link, 500 genes — cluster sizes:", np.bincount(lab_single)[1:])

🔍 **What just happened?** The headline result is **not** robust to every choice:

- **Single link** never finds ALL vs AML: its 2-cluster cut always splits off **one** patient (sizes 71 + 1 — printed
  above for 500 genes; the same happens for every gene number, hence ARI ≈ 0 in every row) — the
  chaining/outlier behaviour of Sections 6.4 and 7.2 on real data.
- **Average** and **complete** link swing between moderate (ARI ≈ 0.4–0.5) and excellent (≈ 0.9) agreement depending on
  how many genes are kept; with 500 genes average link happens to do very well, but with 250 it does not.
- **k-means** is the most stable here (ARI ≈ 0.8–0.9 throughout).

The honest summary is: *the dominant structure of these data is related to ALL vs AML, and most reasonable pipelines
find it approximately* — not "clustering recovers the diagnosis perfectly". If we had tried pipelines until the
agreement looked best and reported only that one, we would be fooling ourselves (and our readers).

⚠️ **Clusters are hypotheses, not discoveries.** In a new data set without labels, a clean two-cluster split could just
as well reflect **batch effects** — which lab processed the samples, bone marrow vs peripheral blood, RNA quality — as
biology. Validation means checking clusters against independent information (clinical outcome, another cohort, known
markers) — the "external criteria" of [Module 13](13_mixture_models_em.ipynb).

✏️ **Your turn.** (a) Replace `metric="correlation"` by `metric="euclidean"` in cell 8.1 (keep the centring) and re-run
8.1–8.2: does average link still find ALL/AML? (b) Cluster the patients into $K = 3$ groups with `fcluster(..., 3, ...)`
and look at the contingency table — which diagnosis gets split? (In the original study, ALL included B-cell and T-cell
cases; our data set does not carry those sub-labels, so treat any sub-split as a hypothesis.)

### 📑 Slide connection — `006_clustering`, slides 56–57

- **What the slides show.** The lotus expression heatmap reappears (56): genes clustered by their expression across plant parts, with a
    dendrogram on top. "What you should know": clustering as unsupervised learning; the types of clustering algorithms;
    the assumptions each makes and what we get from them; unsolved issues — number of clusters, initialisation (57).
- **What this notebook adds.** A full class-discovery analysis on the real Golub data with a pre-declared pipeline, our own tree checked against
    SciPy, a two-way clustered heatmap, a contingency table and ARI against the diagnoses, and a sensitivity table showing
    that the answer depends on the number of genes and the linkage — the "unsolved issues" of slide 57 made concrete.
- **Exam tip.** When asked to "interpret" a clustering result, say what the clusters were compared with (external labels?),
    how robust they are to choices (distance, linkage, K, gene filter) and what else could explain them (batch effects).

## 📌 Key takeaways

- **Clustering is unsupervised learning**: there is only $X$, no labels; we look for groups with high within-cluster and
  low between-cluster similarity, and we can cluster samples, features (genes), or both.
- **The distance is part of the model.** Euclidean distance compares absolute levels; correlation compares shapes. For
  co-expression, z-score each gene (then Euclidean distance ≡ correlation distance).
- **k-means** minimises $\sum_k\sum_{i\in C_k}\lVert\mathbf x_i-\boldsymbol\mu_k\rVert^2$ by alternating *assign to the
  nearest centre* and *move centres to cluster means*; neither step can increase the objective, so it converges — but only
  to a **local optimum**: use many restarts (or k-means++).
- **Choosing K is unsolved**; the elbow of the objective helps on clean data and is often uninformative on real data.
- **Hierarchical clustering** merges the closest clusters bottom-up; **single link** (min) chains, **complete link** (max)
  makes compact clusters, **average link** is the robust default. The dendrogram's heights are the merge distances.
- Hierarchical clustering needs **no K** and exposes **outliers**, but costs at least $O(n^2)$ time and memory, merges
  greedily, and the cut is **subjective**.
- On real data (Golub), unsupervised clustering largely rediscovers ALL vs AML, but the agreement depends on analysis
  choices — clusters are **hypotheses** to be validated.
- Both methods are **non-probabilistic**; [Module 13](13_mixture_models_em.ipynb) gives k-means a probabilistic cousin —
  the Gaussian mixture model — with soft assignments and a likelihood.

---
## ✅ Checkpoint

1. Genes P = (1, 2, 3, 4) and Q = (2, 4, 6, 8) are measured at four time points. Compute their Euclidean distance and their correlation distance. Which measure says they are co-regulated?
   <details><summary>Answer</summary>Euclidean: $\sqrt{1^2+2^2+3^2+4^2}=\sqrt{30}\approx 5.48$. Q = 2P, so $r = 1$ and $1-r = 0$: correlation says they have identical shapes (co-regulated); Euclidean says they differ.</details>
2. One k-means iteration by hand: 1-D data {1, 2, 3, 8, 9, 10}, starting centres μ₁ = 1, μ₂ = 3. Give the assignments, the new centres and the objective before and after the update.
   <details><summary>Answer</summary>Nearest centre: 1, 2 → μ₁ (2 is at distance 1 from both; ties go to the first centre); 3, 8, 9, 10 → μ₂. Objective with old centres: 0 + 1 + 0 + 25 + 36 + 49 = 111. New centres: μ₁ = 1.5, μ₂ = 7.5. Objective: 0.25 + 0.25 + 20.25 + 0.25 + 2.25 + 6.25 = 29.5. The next assignment moves 3 to μ₁, giving centres 2 and 9 and objective 4.</details>
3. Why can the update step of k-means never increase the objective?
   <details><summary>Answer</summary>With memberships fixed, each cluster's contribution $\sum_{i\in C_k}\lVert x_i-\mu\rVert^2$ is minimised by $\mu$ = the mean of the members (derivative $-2\sum (x_i-\mu)=0$). Setting the centre to the mean is therefore the best possible choice, so the objective can only go down or stay the same.</details>
4. Your k-means objective is 120 in one run and 95 in another run with the same K on the same data. What happened, and which run do you keep?
   <details><summary>Answer</summary>The runs started from different initial centres and converged to different local optima. Keep the run with the lower objective (95); in practice run many restarts (scikit-learn `n_init`) or use k-means++ seeding.</details>
5. Using slide 36's matrix (Edna, Lisa, Marge, Selma, Patty), perform complete-link clustering. Give the merge heights.
   <details><summary>Answer</summary>Selma+Patty at 1; Lisa+Marge at 2; {Lisa, Marge} vs {Selma, Patty}: max{4, 4, 3, 3} = 4; Edna joins at max{8, 8, 7, 7} = 8. Heights: 1, 2, 4, 8 (single link gave 1, 2, 3, 7).</details>
6. Clusters A = {a₁, a₂} and B = {b₁} with d(a₁, b₁) = 2 and d(a₂, b₁) = 6. What are the single, complete and average link distances between A and B?
   <details><summary>Answer</summary>Single = min = 2; complete = max = 6; average = (2 + 6)/2 = 4.</details>
7. In `agglomerative`, what does the line `new = (size[a] * dist[a] + size[b] * dist[b]) / (size[a] + size[b])` implement, and why are the sizes needed?
   <details><summary>Answer</summary>The average-link update: the average over all cross-pairs between $A\cup B$ and $C$ equals the size-weighted average of $d(A,C)$ and $d(B,C)$. Without the weights a large cluster and a singleton would count equally, which is not the average over all pairs.</details>
8. Single linkage on the Golub data cut into two clusters gives sizes 71 and 1. What does that tell you?
   <details><summary>Answer</summary>Single link joins clusters through their closest members, so the most isolated patient (an outlier) is the last to join and forms its own cluster at the top of the tree; the 2-cluster cut therefore isolates an outlier instead of finding the main ALL/AML structure (chaining).</details>

## 🧠 Questions & discussion

1. Eisen *et al.* clustered genes with correlation distance. Give an example of a biological question where you would prefer Euclidean distance on raw values instead.
2. k-means assumes compact, roughly spherical clusters of similar size. Sketch a biological data set where this assumption fails (think of cell-differentiation trajectories, or one rare cell type) and suggest a better method.
3. The Golub clusters agree with the diagnoses. Suppose instead they had agreed with the hospital that sent the samples. What would you conclude, and what would you do next?
4. Hierarchical clustering returns a tree, k-means a flat partition. When is a tree a more honest description of biology than a partition (hint: cell types and subtypes)?
5. We fixed the analysis pipeline before looking at the labels. Why does that matter, and how is it related to selection bias in classification (Ambroise & McLachlan 2002; [Module 23](23_model_selection.ipynb))?

## 🚀 Bonus — going further

- **Divisive clustering.** Implement a top-down variant: start with all objects, split the cluster with the largest objective in two with k-means, repeat ("bisecting k-means"). Compare its tree with average linkage on the time course.
- **k-medoids.** Replace the mean by the *medoid* (the member with the smallest total distance to the others). Now any distance, including 1 − r, can be used directly. Implement it and run it on the time-course genes with correlation distance.
- **Stability.** Re-run the Golub analysis on 100 bootstrap resamples of the patients and record how often each pair of patients ends up in the same cluster (a consensus matrix). Which patients are unstable?
- **Probabilistic clustering.** [Module 13](13_mixture_models_em.ipynb) turns k-means into a Gaussian mixture model with soft assignments; the sister course's Module 04 applies GMMs, k-means, hierarchical clustering and the silhouette to cryo-electron tomography particles: [`../../../../cryoem/tutorial/BiologyToResearch/`](../../../../cryoem/tutorial/BiologyToResearch/).
- **Related tutorials in this repository.** Generative vs discriminative classification of breast-cancer data: [`../../GenerativeDiscriminativeClassification/`](../../GenerativeDiscriminativeClassification/); imaging AI: [`../../../../sci_img/tutorial/README.md`](../../../../sci_img/tutorial/README.md).

## 📝 Lab note

Copy this template into your lab notebook and fill it in.

```text
Module: 12 — Clustering: k-means and hierarchical clustering        Date:
Input: SIMULATED 2-D toys and a yeast-like time course (300 genes × 12 times); slide distance matrices; Golub 72 × 3,571 (real)
Output: cluster labels, k-means objective curves, dendrograms, heatmap, ARI vs ALL/AML
Method: k-means (Lloyd, many restarts, k-means++); agglomerative clustering (single / complete / average link)
Key equation: Obj = Σ_k Σ_{i∈C_k} ‖x_i − μ_k‖²;  d_single = min, d_complete = max, d_average = mean over cross-pairs
Key parameters: K; distance (Euclidean vs 1 − r); linkage; number of genes kept; number of restarts
What changed when I modified …: (e.g. z-scoring genes; noise level; number of genes; linkage)
One failure mode: (e.g. single-link chaining isolates one patient; k-means local optimum)
Connection to 02-620 slides: 006_clustering slides 2–57 (k-means 20–33; hierarchical 35–55)
```

## 📚 References

- Carnegie Mellon University 02-620 *Machine Learning for Scientists*, Spring 2025 lecture slides, deck `006_clustering` (course design S. Kim; instructors M. Xu and M. Zhang). Also deck `010_mixture-model_mz_031725`, slides 3 and 78.
- Hastie T., Tibshirani R., Friedman J. (2009) *The Elements of Statistical Learning*, 2nd ed., §14.3 (cluster analysis: k-means, hierarchical clustering). Springer.
- Murphy K. P. (2012) *Machine Learning: A Probabilistic Perspective*, §11.4.2.5 (k-means) and §25.5 (hierarchical clustering). MIT Press.
- Eisen M. B., Spellman P. T., Brown P. O., Botstein D. (1998) Cluster analysis and display of genome-wide expression patterns. *PNAS* 95:14863–14868.
- Spellman P. T. *et al.* (1998) Comprehensive identification of cell cycle–regulated genes of the yeast *Saccharomyces cerevisiae* by microarray hybridization. *Mol Biol Cell* 9:3273–3297.
- Golub T. R. *et al.* (1999) Molecular classification of cancer: class discovery and class prediction by gene expression monitoring. *Science* 286:531–537.
- Dudoit S., Fridlyand J., Speed T. P. (2002) Comparison of discrimination methods for the classification of tumors using gene expression data. *JASA* 97:77–87.
- Perou C. M. *et al.* (2000) Molecular portraits of human breast tumours. *Nature* 406:747–752.
- Arthur D., Vassilvitskii S. (2007) k-means++: the advantages of careful seeding. *Proc. 18th ACM-SIAM Symposium on Discrete Algorithms (SODA)*, 1027–1035.
- Ambroise C., McLachlan G. J. (2002) Selection bias in gene extraction on the basis of microarray gene-expression data. *PNAS* 99:6562–6566.

## ➡️ Next

k-means forces every sample into exactly one cluster, even a sample sitting right on the border. In
[Module 13 · Mixture models and the EM algorithm](13_mixture_models_em.ipynb) we replace hard assignments by
probabilities, write clustering as a probability model, and learn it with the EM algorithm — of which k-means turns out
to be a limiting case.